Here is an XML parsing project that came out of a discussion on the 'gltreebank' listserv. There was a request for "an *Index Locorum* to Smyth's *Greek Grammar*" and one was identified quickly—W. A. Schumann's 1961 *Index of passages cited in Herbert Weir Smyth, Greek grammar.* (GRBS Scholarly Aids 1). [Here](https://catalog.hathitrust.org/Record/001811341) is a link to the item—with full text—in HathiTrust. I often use the Perseus Digital Library edition of [Smyth](http://www.perseus.tufts.edu/hopper/text?doc=Perseus%3Atext%3A1999.04.0007%3Asmythp%3D1) and remembered that citations in this text are linked to their Perseus sources. With this in mind, I decided to recompile something like Schumann's index by parsing the Perseus XML.

The workflow is as follows: (1) compile a list of Smyth XML chunks from the Perseus TOC (provided in the left column of the Smyth HTML); (2) parse the chunks for `<bibl>` elements (the element used for citations), with annotations for `<author>` and `<title>` where applicable; (3) put the data into more user-friendly formats—a Pandas DataFrame and a CSV. The result is a compilation of passages per Smyth chapter. This can be sorted and reindexed to more closely approximate Schumann's index (cf. the `df_auth` DataFrame and corresponding CSV file).

Note a small data-cleanup detail: the encoding for Homeric references in this edition (1) does not include the author, and (2) refers to books by their Greek-letter indices, e.g. 'Α' for *Iliad* 1 and 'α' for *Odyssey* 1. These are corrected both in the parser and with helper functions below. References to *C.I.A.* and *I.G.A.* have been manually corrected—at some point I should correct the annotations in the XML itself and submit a PR.

In [ ]:
# Imports
import urllib.request
from lxml import etree

from collections import defaultdict
import natsort as ns

import pandas as pd

import time

from pprint import pprint

In [ ]:
# Constants
perseus_xml_base_url = "http://www.perseus.tufts.edu/hopper/xmlchunk?doc="
smyth_toc_url = "http://www.perseus.tufts.edu/hopper/xmltoc?doc=Perseus%3Atext%3A1999.04.0007%3Asmythp%3D1"

# The Hopper XML endpoints reject default urllib/curl user-agents; a
# browser-style UA is enough to get a 200.
UA = "Mozilla/5.0 (compatible; ExploratoryPhilologyBlog/1.0; +https://exploratoryphilology.com)"


def fetch(url):
    req = urllib.request.Request(url, headers={'User-Agent': UA})
    with urllib.request.urlopen(req) as f:
        return f.read()

## Getting Smyth XML chunks from the Perseus TOC

In [ ]:
# Fetch the TOC and extract the chunk refs
perseus_toc_xml = fetch(smyth_toc_url)
root = etree.fromstring(perseus_toc_xml)

chapters = root.findall('.//chunk')
refs = [chapter.attrib['ref'] for chapter in chapters]

print(f'{len(refs)} chunks listed in the Smyth TOC')

## Parse Smyth XML chunks for citations

The cell below walks every chunk in the TOC and pulls each `<bibl>` element. With ~3,000 chunks and a polite 250 ms delay between requests, this takes a few minutes on first run. Quarto's `freeze: true` is set for this post, so the work is cached in `_freeze/` and the network round-trips only happen once.

In [ ]:
def get_smyth_xmls(refs, delay=0.25):
    for ref in refs:
        time.sleep(delay)
        yield fetch(perseus_xml_base_url + ref)

In [ ]:
citations = []

for i, xml in enumerate(get_smyth_xmls(refs)):
    root = etree.fromstring(xml)
    milestone = root.find('.//milestone')
    smyth_id = milestone.attrib['id']
    bibls = root.findall('.//bibl')
    for bibl in bibls:
        cit, author_, title_, loc = None, None, None, None
        if 'n' in bibl.attrib.keys():
            cit = bibl.attrib['n']
        else:
            cit = None
        if bibl.find('author') is not None:
            author = bibl.find('author')
            author_ = author.text
        else:
            author = None
        if bibl.find('title') is not None:
            title = bibl.find('title')
            if title.xpath('foreign'):  # Handle Homer
                author_ = 'H.'
                title_ = title.find('foreign').text
            else:
                title_ = title.text
            loc = title.tail
        else:
            title = None
            loc = author.tail
        citations.append((smyth_id, cit, author_, title_, loc))

print(f'Parsed {len(citations)} citations from {len(refs)} Smyth chunks')

## Organize citations in a DataFrame

In [ ]:
df = pd.DataFrame(citations, columns=['smyth-id', 'citation', 'author', 'work', 'loc'])

In [ ]:
# Helper functions for fixing Homeric citations

def fix_homeric_citation(letter, cit):
    if letter:
        letters = [l for l in 'ΑΒΓΔΕΖΗΘΙΚΛΜΝΞΟΠΡΣΤΥΦΧΨΩ']
        if letter.upper() in letters:
            return f'{letters.index(letter.upper()) + 1}.{cit.strip()}'
    return cit


def get_homeric_work(letter):
    if letter:
        letters = [l for l in 'ΑΒΓΔΕΖΗΘΙΚΛΜΝΞΟΠΡΣΤΥΦΧΨΩ']
        if letter.upper() in letters:
            if letter.isupper():
                return 'Il.'
            else:
                return 'Od.'
    return letter

In [ ]:
# Manually add CIA & IGA entries (not currently annotated correctly in Perseus)
# TODO: Correct entry in xml and make PR?

df.loc[(df['smyth-id'] == 's904') & (df['work'] == 'C.I.A.') & (df['loc'] == ' /lref>'), 'loc'] = '4.2.59b'
df.loc[(df['smyth-id'] == 's1473') & (df['work'] == 'C.I.A.') & (df['loc'] == ' /lref>'), 'loc'] = '2, add. 834 b, 1, 38'
df.loc[(df['smyth-id'] == 's1488') & (df['work'] == 'C.I.A.') & (df['loc'] == ' /lref>'), 'loc'] = '2.55.9'
df.loc[(df['smyth-id'] == 's1527') & (df['work'] == 'C.I.A.') & (df['loc'] == ' /lref>'), 'loc'] = '2.17.7'
df.loc[(df['smyth-id'] == 's1923') & (df['work'] == 'C.I.A.') & (df['loc'] == ' /lref>'), 'loc'] = '1.32'
df.loc[(df['smyth-id'] == 's1923') & (df['work'] == 'I.G.A.') & (df['loc'] == ' /lref>'), 'loc'] = '348'

In [ ]:
# Natsort df by Smyth ids

df['smyth-id'] = pd.Categorical(df['smyth-id'], ordered=True, categories=ns.natsorted(df['smyth-id'].unique()))
df = df.sort_values('smyth-id')
df['loc'] = df[['work', 'loc']].apply(lambda x: fix_homeric_citation(*x), axis=1)
df['work'] = df['work'].apply(lambda x: get_homeric_work(x))

In [ ]:
df

In [ ]:
# Natsort by author/work

df_auth = df.copy()
df_auth['loc'] = pd.Categorical(df_auth['loc'], ordered=True, categories=ns.natsorted(df_auth['loc'].unique()))
df_auth = df_auth.sort_values(['author', 'work', 'loc'])
df_auth = df_auth.reset_index(drop=True)
df_auth

## Export DataFrames to CSV

In [ ]:
df.to_csv('data/smyth_citations.csv', index=False)
df_auth.to_csv('data/smyth_citations_by_author.csv', index=False)

## Stats on the Perseus-Smyth citations

In [ ]:
total_citations = len(df['smyth-id'])
unique_authors = len(set(df['author']))
unique_works = len(set([''.join(filter(None, (item[0], item[1])))
                        for item in zip(df['author'], df['work'])]))
freq_author = df['author'].value_counts().keys()[0]
freq_work = ' '.join(df.groupby(['author', 'work']).size().idxmax())

print('Perseus-Smyth stats')
print('-' * 20)
print(f'Total citations: {total_citations}')
print(f'Unique authors: {unique_authors}')
print(f'Unique works: {unique_works}')
print(f'Most frequent author: {freq_author}')
print(f'Most frequent work: {freq_work}')

In [ ]:
print('Perseus-Smyth top authors')
print('-' * 20)
for item in list(df['author'].value_counts().items())[:10]:
    print(f'{item[0]} ({item[1]})')

In [ ]:
print('Perseus-Smyth top works')
print('-' * 20)
df['work_'] = df['work'].astype(str)
for item in list(df.groupby(['author', 'work_']).size().nlargest(10).items()):
    if item[0][1] == 'None':
        print(f'{item[0][0]} ({item[1]})')
    else:
        print(f'{item[0][0]}, {item[0][1]} ({item[1]})')